In [ ]:
import pandas as pd 

excel_data = pd.read_excel("./IPL.xls")
sort_in_descending = excel_data.sort_values(by='Year', kind='quicksort')

excel_data.isnull().sum()

excel_data.drop_duplicates(inplace=True)

print(sort_in_descending.shape)
print(sort_in_descending.info())
print(sort_in_descending.isnull().sum())
# print(sort_in_descending)

(514, 43)
<class 'pandas.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Match_Number                514 non-null    int64  
 1   Match                       514 non-null    str    
 2   Date                        514 non-null    str    
 3   Year                        514 non-null    int64  
 4   Venue                       514 non-null    str    
 5   Team_Batting_First          514 non-null    str    
 6   Team_Batting_Second         514 non-null    str    
 7   Bat_First_Runs_Scored       514 non-null    int64  
 8   Bat_First_Wickets_Lost      514 non-null    int64  
 9   Bat_First_Overs_Consumed    514 non-null    float64
 10  Bat_First_Overs_Allocated   514 non-null    int64  
 11  Bat_Second_Runs_Scored      514 non-null    int64  
 12  Bat_Second_Wickets_Lost     514 non-null    int64  
 13  Bat_Second_Overs_Consumed   514 non-

In [ ]:
# most successful team

team_wins = excel_data['Winner'].value_counts()
team_win_bat = excel_data['Winning_Team'].value_counts()

most_wins_teams = team_wins.idxmax()
most_wins_count = team_wins.max()

print("Team with most wins:", most_wins_teams)
print("Total wins:", most_wins_count)
print(team_win_bat)


Team with most wins: Chennai
Total wins: 79
Winning_Team
Chasing         272
FirstBatting    236
Match Tied        6
Name: count, dtype: int64


In [21]:

#how venue effects the game and which stadium has higest scoring
most_venue = excel_data['Venue'].value_counts()

print("\n", most_venue.sort_values(ascending=False))


 Venue
Mumbai            62
Chennai           48
Bangalore         47
Kolkata           47
Delhi             45
Mohali            35
Hyderabad         34
Jaipur            33
Pune              21
Durban            15
Centurion         12
Ahmedabad         12
Mumbai (BS)       11
Dharamsala         9
Johannesburg       8
Cape Town          7
Port Elizabeth     7
Cuttack            7
Ranchi             7
Abu Dhabi          7
Dubai              7
Sharjah            6
Kochi              5
Visakhapatnam      5
Raipur             4
East London        3
Kimberley          3
Nagpur             3
Bloemfontein       2
Indore             2
Name: count, dtype: int64


In [51]:
# find which match number created the highest run rate batting first(column T) ? which team batted first and how many runs they scored in how many overs

highest_rr = excel_data.loc[excel_data['Bat_First_Run_Rate'].idxmax()]

top_rr = excel_data.sort_values(by='Bat_First_Run_Rate', ascending=False)

print("Match Number:", highest_rr['Match_Number'])
print("Team Batting First:", highest_rr['Team_Batting_First'])
print("Runs Scored:", highest_rr['Bat_First_Runs_Scored'])
print("Overs Played:", highest_rr['Bat_First_Overs_Consumed'])
print("Run Rate:", highest_rr['Bat_First_Run_Rate'])
print("top 10 highest run rate\n", top_rr['Bat_First_Run_Rate'].head(10))


Match Number: 391
Team Batting First: Bangalore
Runs Scored: 106
Overs Played: 8.0
Run Rate: 13.25
top 10 highest run rate
 390    13.250000
351    13.150000
146    12.300000
506    12.272727
1      12.000000
500    11.750000
236    11.600000
199    11.550000
425    11.550000
455    11.300000
Name: Bat_First_Run_Rate, dtype: float64


In [58]:
#total matches per year
total_matches = excel_data.groupby('Year').size()

#batting first won
bat_first_won = excel_data[excel_data['Winning_Team']=='FirstBatting'].groupby('Year').size()

#chasing won
chasing_won = excel_data[excel_data['Winning_Team']=='Chasing'].groupby('Year').size()

match_tied = excel_data[excel_data['Winning_Team'] == 'Match Tied'].groupby('Year').size()

result = pd.DataFrame({
    'Total Matches': total_matches,
    'Batting First Won': bat_first_won,
    'Chasing Won': chasing_won,
    'Match Tied': match_tied
})

result = result.fillna(0)

result = result.astype(int)

print(result)


      Total Matches  Batting First Won  Chasing Won  Match Tied
Year                                                           
2008             58                 22           36           0
2009             57                 26           30           1
2010             60                 31           28           1
2011             72                 32           40           0
2012             74                 34           40           0
2013             76                 37           37           2
2014             60                 22           37           1
2015             57                 32           24           1


In [ ]:
#Break out year by year how many close matches were played
import pandas as pd

excel_data = pd.read_excel("./IPL.xls")

#  wickets remaining for chasing team (Bat_Second)
#  each team has 10 wickets available
wickets_remaining = 10 - excel_data['Bat_Second_Wickets_Lost']

close_matches = excel_data.loc[
    (excel_data['Winning_Team'] == 'Match Tied') |                           
    (excel_data['Balls_Remaining'] <= 3) |                                     
    ((excel_data['Winning_Team'] == 'Chasing') & (wickets_remaining <= 1)) | 
    (
        (excel_data['Winning_Team'] == 'FirstBatting') &
        (excel_data['Win_Type']     == 'run') &          
        (excel_data['Winning_Margin'] <= 5)
    )                                                             
]


result = (
    close_matches
    .groupby('Year')
    .size()
    .reset_index()
    .rename(columns={0: 'No. of Close Matches'})
)


print(result.to_string(index=False))

 Year  No. of Close Matches
 2008                    28
 2009                    33
 2010                    28
 2011                    35
 2012                    46
 2013                    39
 2014                    33
 2015                    39


In [ ]:
# For the top 4 teams, what percentage of their wins came while batting first?

# wins while batting first for each team
batting_first_wins = excel_data[excel_data['Winning_Team'] == 'FirstBatting'].groupby('Winner').size()

# total wins for each team
team_wins = excel_data['Winner'].value_counts()

# top 4 teams
top_4_teams = team_wins.head(4)

# percentage for each team
percent_first_bat = (batting_first_wins[top_4_teams.index] / top_4_teams) * 100

print("Percentage of wins while batting first (for top 4 teams):")
print(percent_first_bat)
print("\nInterpretation: Out of total wins, what % came while batting first")

Percentage of wins while batting first (for top 4 teams):
Winner
Chennai      56.962025
Mumbai       56.164384
Rajasthan    37.704918
Kolkata      41.666667
dtype: float64

Interpretation: Out of total wins, what % came while batting first


In [22]:
# Find matches where batting second had 6 wickets in hand at 15 overs, 
# needed 33 runs in last 5 overs, but still lost

# 6 wickets in hand = 4 wickets lost (10 - 4 = 6)
wickets_in_hand_at_15 = 10 - excel_data['Bat_Second_15_ov_wkts_lost']

# Runs needed in last 5 overs = Target - Score at 15 overs
target = excel_data['Bat_First_Runs_Scored'] + 1  # Need to beat the first batting score
runs_needed_last_5 = target - excel_data['Bat_Second_15_ov_score']

# Filter matches with the criteria
matching_matches = excel_data[
    (wickets_in_hand_at_15 == 6) & 
    (runs_needed_last_5 == 33) & 
    (excel_data['Winning_Team'] != 'Chasing')
]

print(f"Found {len(matching_matches)} match(es) with the criteria:\n")
print(matching_matches[['Match_Number', 'Date', 'Team_Batting_First', 'Team_Batting_Second', 
                        'Bat_First_Runs_Scored', 'Bat_Second_Runs_Scored', 
                        'Bat_Second_15_ov_score', 'Bat_Second_15_ov_wkts_lost',
                        'Winner']])


Found 1 match(es) with the criteria:

     Match_Number    Date Team_Batting_First Team_Batting_Second  \
342           343  Apr 17          Hyderabad                Pune   

     Bat_First_Runs_Scored  Bat_Second_Runs_Scored  Bat_Second_15_ov_score  \
342                    119                     108                    87.0   

     Bat_Second_15_ov_wkts_lost     Winner  
342                         4.0  Hyderabad  


In [23]:
wickets_in_hand_at_15 = 10 - excel_data['Bat_First_15_ov_wkts_lost']
runs_last_5_ov = excel_data['Bat_First_Runs_Scored'] - excel_data['Bat_First_15_ov_score']


matching_matches = excel_data[
    (wickets_in_hand_at_15 == 2) & 
    (runs_needed_last_5 < 25) 
]
print(f"Found {len(matching_matches)} match(es) with the criteria:\n")

print(matching_matches[['Match_Number', 'Date', 'Team_Batting_First', 'Team_Batting_Second', 
                        'Bat_First_Runs_Scored', 'Bat_Second_Runs_Scored', 
                        'Bat_Second_15_ov_score', 'Bat_Second_15_ov_wkts_lost',
                        'Winner']])


Found 0 match(es) with the criteria:

Empty DataFrame
Columns: [Match_Number, Date, Team_Batting_First, Team_Batting_Second, Bat_First_Runs_Scored, Bat_Second_Runs_Scored, Bat_Second_15_ov_score, Bat_Second_15_ov_wkts_lost, Winner]
Index: []
